# Week 3.5: High-Accuracy Synthetic-Fact Baseline with Strict Scoring

This notebook trains a fresh LoRA adapter on the Week 3.5 synthetic facts, then evaluates the saved learned adapter on the synthetic forget/retain question sets and the general-control question set using the same strict scorer used in Week 4.

It overwrites the adapter and result reports in:

`/content/drive/MyDrive/Thesis/Week 3.5/results/qwen05_high_accuracy_baseline`

## 0. Runtime Setup

In Colab, use `Runtime -> Restart session`, then select a T4 GPU runtime before running all cells.

In [ ]:
%pip uninstall -q -y bitsandbytes
%pip install -q -U "transformers==4.48.3" "accelerate==1.3.0" "peft==0.19.1" "datasets==3.2.0" sentencepiece "pandas==2.2.3"
%pip install -q --no-deps "bitsandbytes==0.49.2"

## 1. Mount Google Drive and Set Paths

In [ ]:
# GitHub-backed Colab setup. Keep GITHUB_TOKEN in Colab Secrets.
from pathlib import Path
import sys
import urllib.request

HELPER_PATH = Path('/content/github_colab_sync.py')
HELPER_URL = 'https://raw.githubusercontent.com/HannanSeyfi/unlearning-thesis/main/Tools/github_colab_sync.py'
if not HELPER_PATH.exists():
    urllib.request.urlretrieve(HELPER_URL, HELPER_PATH)
if str(HELPER_PATH.parent) not in sys.path:
    sys.path.insert(0, str(HELPER_PATH.parent))

from github_colab_sync import commit_and_push, resolve_week35_baseline_dir, setup_colab_repo

THESIS_DIR = setup_colab_repo()

WEEK35_DIR = THESIS_DIR / 'Week 3.5'
DATA_DIR = WEEK35_DIR / 'data' / 'synthetic_facts_v1'
CONTROL_DIR = WEEK35_DIR / 'data' / 'general_controls_v1'
OUTPUT_DIR = WEEK35_DIR / 'results' / 'qwen05_high_accuracy_baseline'

TRAIN_PATH = DATA_DIR / 'train_all.jsonl'
EVAL_FORGET_PATH = DATA_DIR / 'eval_forget.jsonl'
EVAL_RETAIN_PATH = DATA_DIR / 'eval_retain.jsonl'
GENERAL_CONTROL_PATH = CONTROL_DIR / 'general_control.jsonl'

ADAPTER_DIR = OUTPUT_DIR / 'adapter'
TRAINER_CHECKPOINT_DIR = OUTPUT_DIR / 'trainer_checkpoints'
BASE_RESULTS_CSV_PATH = OUTPUT_DIR / 'base_model_results.csv'
LORA_RESULTS_CSV_PATH = OUTPUT_DIR / 'lora_model_results.csv'
BASE_GENERAL_CSV_PATH = OUTPUT_DIR / 'base_general_control_results.csv'
LORA_GENERAL_CSV_PATH = OUTPUT_DIR / 'lora_general_control_results.csv'
COMPARISON_CSV_PATH = OUTPUT_DIR / 'base_vs_lora_comparison.csv'
ALL_TEST_RESULTS_CSV_PATH = OUTPUT_DIR / 'all_test_results.csv'
PERCENTAGE_SUMMARY_CSV_PATH = OUTPUT_DIR / 'percentage_summary.csv'
PERCENTAGE_BY_CATEGORY_CSV_PATH = OUTPUT_DIR / 'percentage_by_category.csv'
METRICS_PATH = OUTPUT_DIR / 'metrics.json'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
TRAINER_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for path in [TRAIN_PATH, EVAL_FORGET_PATH, EVAL_RETAIN_PATH, GENERAL_CONTROL_PATH]:
    assert path.exists(), f'Missing file: {path}'

print('Training file:', TRAIN_PATH)
print('Forget eval file:', EVAL_FORGET_PATH)
print('Retain eval file:', EVAL_RETAIN_PATH)
print('General-control file:', GENERAL_CONTROL_PATH)
print('Output folder to overwrite:', OUTPUT_DIR)


## 2. Imports and Configuration

In [ ]:
import gc
import json
import random
import re
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
USE_4BIT = True
MAX_LENGTH = 192
MAX_NEW_TOKENS = 16
NUM_EPOCHS = 20
LEARNING_RATE = 3e-4
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
TRAIN_ON_FACT_VALUE_ONLY = True
ADD_EVAL_PROMPTS_TO_TRAINING = False
SYNTHETIC_SYSTEM_PROMPT = 'You answer questions about fictional synthetic people using the provided learned facts.'
GENERAL_SYSTEM_PROMPT = 'Answer the question concisely. Return only the requested answer without explanation.'

assert ADD_EVAL_PROMPTS_TO_TRAINING is False
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('Selected model:', MODEL_ID)
print('Use 4-bit loading:', USE_4BIT)
print('Strict scorer max_new_tokens:', MAX_NEW_TOKENS)
print('Evaluation prompts added to training:', ADD_EVAL_PROMPTS_TO_TRAINING)

## 3. Load Week 3.5 Data

In [ ]:
def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = read_jsonl(TRAIN_PATH)
eval_forget_records = read_jsonl(EVAL_FORGET_PATH)
eval_retain_records = read_jsonl(EVAL_RETAIN_PATH)
general_control_records = read_jsonl(GENERAL_CONTROL_PATH)

train_prompt_keys = {
    (record['entity_id'], record['fact_type'], record['prompt'].strip().lower())
    for record in train_records
}

def prompt_was_seen(record):
    key = (
        record.get('entity_id'),
        record.get('fact_type'),
        record['prompt'].strip().lower(),
    )
    return key in train_prompt_keys

num_seen_eval = sum(
    prompt_was_seen(record)
    for record in eval_forget_records + eval_retain_records
)
num_eval = len(eval_forget_records) + len(eval_retain_records)

print('Training examples:', len(train_records))
print('Forget eval examples:', len(eval_forget_records))
print('Retain eval examples:', len(eval_retain_records))
print('Evaluation prompts identical to training prompts:', num_seen_eval)
print('Held-out paraphrase prompts:', num_eval - num_seen_eval)
print('Fixed general-control questions:', len(general_control_records))

pd.DataFrame(train_records)[['example_id', 'split', 'prompt', 'answer', 'fact_value']].head()

## 4. Tokenizer, Model Loader, and Strict Evaluation Helpers

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def load_base_model():
    if USE_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        return AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=quantization_config,
            device_map='auto',
        )
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )

def chat_text(messages, add_generation_prompt=False):
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    text = ''
    for message in messages:
        role = message['role'].capitalize()
        text += f'{role}: {message["content"]}\n'
    if add_generation_prompt:
        text += 'Assistant:'
    return text

def normalize_text(text):
    text = unicodedata.normalize('NFKD', str(text))
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r'\s+', ' ', text.strip().lower())
    return text.strip(' .,!?:;"\'')

def contains_value(generated, expected):
    generated = normalize_text(generated)
    expected = normalize_text(expected)
    return bool(expected and re.search(rf'(?<!\w){re.escape(expected)}(?!\w)', generated))

def target_answer(record):
    if TRAIN_ON_FACT_VALUE_ONLY and record.get('fact_value'):
        return str(record['fact_value'])
    return str(record.get('answer', record.get('expected_value', '')))

@torch.inference_mode()
def generate_answer(model, prompt, general=False):
    messages = [
        {'role': 'system', 'content': GENERAL_SYSTEM_PROMPT if general else SYNTHETIC_SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text = chat_text(messages, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate_records(model, records, eval_split_name, model_stage, general=False, progress_every=100):
    rows = []
    model.eval()
    for index, record in enumerate(records, start=1):
        expected = target_answer(record)
        generated = generate_answer(model, record['prompt'], general=general)
        rows.append({
            'model_stage': model_stage,
            'eval_split': eval_split_name,
            'prompt_seen_in_training': False if general else prompt_was_seen(record),
            'example_id': record.get('example_id'),
            'entity_id': record.get('entity_id'),
            'full_name': record.get('full_name'),
            'fact_type': record.get('fact_type', record.get('category')),
            'fact_value': expected,
            'prompt': record['prompt'],
            'expected_answer': expected,
            'original_dataset_answer': record.get('answer', record.get('expected_value', expected)),
            'generated_answer': generated,
            'exact_match': normalize_text(generated) == normalize_text(expected),
            'contains_value': contains_value(generated, expected),
        })
        if progress_every and index % progress_every == 0:
            print(f'{model_stage} / {eval_split_name}: {index}/{len(records)}')
    return pd.DataFrame(rows)

def percentage(frame):
    return float(100 * frame['contains_value'].mean())

def score_table(results):
    return (
        results.groupby(['eval_split', 'prompt_seen_in_training'])[
            ['exact_match', 'contains_value']
        ]
        .agg(['mean', 'count'])
        .reset_index()
    )

## 5. Evaluate the Untouched Base Model with Strict Scoring

In [ ]:
base_model = load_base_model()
base_model.config.use_cache = True

base_forget_results_df = evaluate_records(
    base_model, eval_forget_records, 'forget', 'base_before_training'
)
base_retain_results_df = evaluate_records(
    base_model, eval_retain_records, 'retain', 'base_before_training'
)
base_results_df = pd.concat(
    [base_forget_results_df, base_retain_results_df],
    ignore_index=True,
)
base_general_results_df = evaluate_records(
    base_model,
    general_control_records,
    'general',
    'base_before_training',
    general=True,
    progress_every=0,
)

base_results_df.to_csv(BASE_RESULTS_CSV_PATH, index=False)
base_general_results_df.to_csv(BASE_GENERAL_CSV_PATH, index=False)

print('Base-model scores:')
display(score_table(base_results_df))
print('Base general-control contains-value:', f"{percentage(base_general_results_df):.2f}%")
print('Saved base synthetic results:', BASE_RESULTS_CSV_PATH)
print('Saved base general results:', BASE_GENERAL_CSV_PATH)

del base_model
gc.collect()
torch.cuda.empty_cache()

## 6. Prepare Supervised Training Dataset

In [ ]:
def encode_training_record(record):
    messages = [
        {'role': 'system', 'content': SYNTHETIC_SYSTEM_PROMPT},
        {'role': 'user', 'content': record['prompt']},
        {'role': 'assistant', 'content': target_answer(record)},
    ]
    prompt_text = chat_text(messages[:-1], add_generation_prompt=True)
    full_text = chat_text(messages, add_generation_prompt=False)

    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    labels = full['input_ids'].copy()
    prompt_len = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    full['labels'] = labels
    return full

train_dataset = Dataset.from_list(train_records)
train_dataset = train_dataset.map(
    encode_training_record,
    remove_columns=train_dataset.column_names,
)

num_labeled_tokens = [
    sum(label != -100 for label in row['labels'])
    for row in train_dataset
]
assert min(num_labeled_tokens) > 0, 'At least one training example has no answer tokens after truncation.'

print('Encoded training examples:', len(train_dataset))
print('Min labeled tokens:', min(num_labeled_tokens))
print('Max labeled tokens:', max(num_labeled_tokens))

## 7. Train Fresh LoRA Adapter on Synthetic Facts

In [ ]:
training_base_model = load_base_model()
training_base_model.config.use_cache = False
if USE_4BIT:
    training_base_model = prepare_model_for_kbit_training(training_base_model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

model = get_peft_model(training_base_model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(TRAINER_CHECKPOINT_DIR),
    seed=SEED,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_ratio=0.05,
    weight_decay=0.0,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
    ),
)

train_result = trainer.train()
train_metrics = train_result.metrics
train_metrics

## 8. Save the Trained Adapter

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print('Saved fresh strict-baseline adapter to:', ADAPTER_DIR)

del trainer, model, training_base_model
gc.collect()
torch.cuda.empty_cache()

## 9. Reload the Saved Adapter and Evaluate with Strict Scoring

In [ ]:
learned_model = load_base_model()
learned_model = PeftModel.from_pretrained(learned_model, ADAPTER_DIR)
learned_model.config.use_cache = True
learned_model.eval()

lora_forget_results_df = evaluate_records(
    learned_model, eval_forget_records, 'forget', 'lora_after_training'
)
lora_retain_results_df = evaluate_records(
    learned_model, eval_retain_records, 'retain', 'lora_after_training'
)
lora_results_df = pd.concat(
    [lora_forget_results_df, lora_retain_results_df],
    ignore_index=True,
)
lora_general_results_df = evaluate_records(
    learned_model,
    general_control_records,
    'general',
    'lora_after_training',
    general=True,
    progress_every=0,
)

lora_results_df.to_csv(LORA_RESULTS_CSV_PATH, index=False)
lora_general_results_df.to_csv(LORA_GENERAL_CSV_PATH, index=False)

print('LoRA-model scores from reloaded saved adapter:')
display(score_table(lora_results_df))
print('LoRA general-control contains-value:', f"{percentage(lora_general_results_df):.2f}%")
print('Saved LoRA synthetic results:', LORA_RESULTS_CSV_PATH)
print('Saved LoRA general results:', LORA_GENERAL_CSV_PATH)

del learned_model
gc.collect()
torch.cuda.empty_cache()

## 10. Save Comparisons, Summaries, and Metrics

In [ ]:
result_key = [
    'eval_split',
    'prompt_seen_in_training',
    'example_id',
    'entity_id',
    'full_name',
    'fact_type',
    'fact_value',
    'prompt',
    'expected_answer',
]

comparison_df = base_results_df[result_key + [
    'generated_answer', 'exact_match', 'contains_value'
]].merge(
    lora_results_df[result_key + [
        'generated_answer', 'exact_match', 'contains_value'
    ]],
    on=result_key,
    suffixes=('_base', '_lora'),
    validate='one_to_one',
)
comparison_df['contains_value_improved'] = (
    comparison_df['contains_value_lora'].astype(int)
    - comparison_df['contains_value_base'].astype(int)
)
comparison_df['exact_match_improved'] = (
    comparison_df['exact_match_lora'].astype(int)
    - comparison_df['exact_match_base'].astype(int)
)
comparison_df.to_csv(COMPARISON_CSV_PATH, index=False)

def with_test_set(frame, test_set):
    frame = frame.copy()
    frame['test_set'] = test_set
    frame['category'] = frame['fact_type']
    return frame

all_results_df = pd.concat([
    with_test_set(base_results_df, 'synthetic_facts'),
    with_test_set(lora_results_df, 'synthetic_facts'),
    with_test_set(base_general_results_df, 'general_control'),
    with_test_set(lora_general_results_df, 'general_control'),
], ignore_index=True)
all_results_df.to_csv(ALL_TEST_RESULTS_CSV_PATH, index=False)

percentage_summary_df = (
    all_results_df.groupby(['model_stage', 'test_set', 'eval_split'], dropna=False)
    .agg(
        num_questions=('contains_value', 'size'),
        num_contains_value_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda values: 100 * values.mean()),
        exact_match_percentage=('exact_match', lambda values: 100 * values.mean()),
    )
    .reset_index()
)
percentage_by_category_df = (
    all_results_df.groupby(['model_stage', 'test_set', 'eval_split', 'category'], dropna=False)
    .agg(
        num_questions=('contains_value', 'size'),
        num_contains_value_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda values: 100 * values.mean()),
        exact_match_percentage=('exact_match', lambda values: 100 * values.mean()),
    )
    .reset_index()
)
percentage_summary_df.to_csv(PERCENTAGE_SUMMARY_CSV_PATH, index=False)
percentage_by_category_df.to_csv(PERCENTAGE_BY_CATEGORY_CSV_PATH, index=False)

display(percentage_summary_df)
print('Saved comparison CSV:', COMPARISON_CSV_PATH)
print('Saved all test results CSV:', ALL_TEST_RESULTS_CSV_PATH)
print('Saved percentage summary CSV:', PERCENTAGE_SUMMARY_CSV_PATH)
print('Saved percentage by category CSV:', PERCENTAGE_BY_CATEGORY_CSV_PATH)

## 11. Write Metrics JSON

In [ ]:
def subset_metrics(results, split, seen):
    subset = results[
        (results['eval_split'] == split)
        & (results['prompt_seen_in_training'] == seen)
    ]
    return {
        'num_examples': int(len(subset)),
        'exact_match': float(subset['exact_match'].mean()),
        'contains_value': float(subset['contains_value'].mean()),
    }

def stage_metrics(results):
    return {
        'forget_all': {
            'num_examples': int((results['eval_split'] == 'forget').sum()),
            'exact_match': float(results.loc[results['eval_split'] == 'forget', 'exact_match'].mean()),
            'contains_value': float(results.loc[results['eval_split'] == 'forget', 'contains_value'].mean()),
        },
        'retain_all': {
            'num_examples': int((results['eval_split'] == 'retain').sum()),
            'exact_match': float(results.loc[results['eval_split'] == 'retain', 'exact_match'].mean()),
            'contains_value': float(results.loc[results['eval_split'] == 'retain', 'contains_value'].mean()),
        },
        'forget_heldout_paraphrases': subset_metrics(results, 'forget', False),
        'retain_heldout_paraphrases': subset_metrics(results, 'retain', False),
        'forget_seen_prompts': subset_metrics(results, 'forget', True),
        'retain_seen_prompts': subset_metrics(results, 'retain', True),
    }

metrics = {
    'run_name': 'week3_5_qwen05_high_accuracy_baseline_strict',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'adapter_dir': str(ADAPTER_DIR),
    'dataset_dir': str(DATA_DIR),
    'evaluation_leakage_prevented': True,
    'train_on_fact_value_only': TRAIN_ON_FACT_VALUE_ONLY,
    'num_train_examples': int(len(train_records)),
    'num_eval_forget_examples': int(len(eval_forget_records)),
    'num_eval_retain_examples': int(len(eval_retain_records)),
    'num_eval_prompts_seen_in_training': int(num_seen_eval),
    'num_general_control_questions': int(len(general_control_records)),
    'num_heldout_paraphrase_prompts': int(num_eval - num_seen_eval),
    'training': {
        'seed': SEED,
        'use_4bit': USE_4BIT,
        'max_length': MAX_LENGTH,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'trainer_metrics': {
            key: float(value)
            for key, value in train_metrics.items()
            if isinstance(value, (int, float))
        },
    },
    'strict_scoring': {
        'source': 'Week 4-compatible scorer',
        'contains_value_rule': 'case-insensitive normalized whole-token regex boundary match',
        'normalization': 'NFKD, remove combining marks, lowercase, normalize whitespace, strip outer punctuation',
        'max_new_tokens': MAX_NEW_TOKENS,
        'synthetic_system_prompt': SYNTHETIC_SYSTEM_PROMPT,
        'general_system_prompt': GENERAL_SYSTEM_PROMPT,
        'learned_model_evaluated_after_save_and_reload': True,
    },
    'base_before_training': stage_metrics(base_results_df),
    'lora_after_training': stage_metrics(lora_results_df),
    'general_control': {
        'base_before_training_contains_value_percentage': float(100 * base_general_results_df['contains_value'].mean()),
        'lora_after_training_contains_value_percentage': float(100 * lora_general_results_df['contains_value'].mean()),
    },
    'files': {
        'base_results_csv': str(BASE_RESULTS_CSV_PATH),
        'lora_results_csv': str(LORA_RESULTS_CSV_PATH),
        'comparison_csv': str(COMPARISON_CSV_PATH),
        'all_test_results_csv': str(ALL_TEST_RESULTS_CSV_PATH),
        'base_general_csv': str(BASE_GENERAL_CSV_PATH),
        'lora_general_csv': str(LORA_GENERAL_CSV_PATH),
        'percentage_summary_csv': str(PERCENTAGE_SUMMARY_CSV_PATH),
        'percentage_by_category_csv': str(PERCENTAGE_BY_CATEGORY_CSV_PATH),
    },
}

METRICS_PATH.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(json.dumps(metrics, indent=2))
print('Saved metrics:', METRICS_PATH)

## 12. Error Samples

In [ ]:
lora_errors_df = lora_results_df[~lora_results_df['contains_value']].copy()
general_errors_df = lora_general_results_df[~lora_general_results_df['contains_value']].copy()

print('LoRA synthetic contains-value errors:', len(lora_errors_df))
if len(lora_errors_df):
    display(
        lora_errors_df.groupby(['eval_split', 'prompt_seen_in_training'])
        .size()
        .reset_index(name='num_errors')
    )
    display(lora_errors_df[[
        'eval_split',
        'prompt_seen_in_training',
        'fact_type',
        'prompt',
        'expected_answer',
        'generated_answer',
        'contains_value',
    ]].head(30))

print('LoRA general-control contains-value errors:', len(general_errors_df))
if len(general_errors_df):
    display(general_errors_df[[
        'fact_type',
        'prompt',
        'expected_answer',
        'generated_answer',
        'contains_value',
    ]].head(30))

In [ ]:
# GitHub output sync
commit_and_push(
    [OUTPUT_DIR],
    'Colab: save Week 3.5 strict baseline outputs',
    repo_dir=THESIS_DIR,
)
